In [0]:
%run ./Classroom-Setup-Common-Python

In [0]:
#####################
# CREATE USER OR SET USER CATALOG
#####################
my_catalog = build_user_catalog(catalog_forced=None)

In [0]:
#####################
# CREATE SCHEMAS
#####################

# DEMO SCHEMA: Create the aibi_dashboards schema inside the user's catalog
my_schema = "aibi_dashboards"
create_schemas(in_catalog=my_catalog, schemas_to_create=[my_schema])

# LAB SCHEMA: Create the aibi_dashboards schema inside the user's catalog
lab_schema = "aibi_dashboards_taxi"
create_schemas(in_catalog=my_catalog, schemas_to_create=[lab_schema])

In [0]:
#####################
# CREATE VOLUMES
#####################


#
# Create the source_files volume inside the aibi_dashboards schema
#
my_volume = "source_files"

spark.sql(f"CREATE VOLUME IF NOT EXISTS `{my_catalog}`.`{my_schema}`.`{my_volume}`")
print(f"✅ Volume ready: {my_catalog}.{my_schema}.{my_volume}")


#
# Create the business_icon volume inside the aibi_dashboards schema
#
my_volume_icon = "business_icon"

spark.sql(f"CREATE VOLUME IF NOT EXISTS `{my_catalog}`.`{my_schema}`.`{my_volume_icon}`")
print(f"✅ Volume ready: {my_catalog}.{my_schema}.{my_volume_icon}")

In [0]:
#################################
## COPY FILES TO VOLUMES
#################################

from pathlib import Path
import time

##
## Set Data Volume Path
##
vol_path = f'/Volumes/{my_catalog}/{my_schema}/{my_volume}'


##
## Get Path for Dashboard Data/Image Import
##
current_path = Path.cwd()
dashboard_data_import_path = current_path.parent / "Includes/data/labs"
dashboard_image_import_path = current_path.parent / "Includes/data/images/business-icon"


#
## FIND DEMO DATA IN WORKSPACE FOLDER
#
try:
    data_path = find_folder('Includes/data/demos')
except:
    data_path = find_folder('data/demos')

copy_workspace_files_to_volume(
    src_workspace_folder=f'{data_path}',
    target_volume_path=vol_path,
    n=4
)

#
## FIND LAB DATA IN WORKSPACE FOLDER
#
try:
    lab_data_path = find_folder('Includes/data/labs')
except:
    lab_data_path = find_folder('data/labs')

copy_workspace_files_to_volume(
    src_workspace_folder=f'{lab_data_path}',
    target_volume_path=vol_path,
    n=1
)

#
## IMAGE ICON PATH
#
try:
    icon_path = find_folder('Includes/business-icon')
except:
    icon_path = find_folder('business-icon')

icon_vol_path = f'/Volumes/{my_catalog}/{my_schema}/{my_volume_icon}'

copy_workspace_files_to_volume(
    src_workspace_folder=f'{icon_path}',
    target_volume_path=icon_vol_path,
    n=1
)

In [0]:
# Set the default catalog and schema so the CREATE TABLE statements below
# don't need three-part names.
spark.sql(f"USE CATALOG `{my_catalog}`")
spark.sql(f"USE SCHEMA  `{my_schema}`")

In [0]:
# ### Build silver tables from the CSVs in the `source_files` volume

# Each statement uses `CREATE TABLE IF NOT EXISTS ... AS SELECT` so the setup is **idempotent** —
# if a table already exists the statement is a no-op. `CAST()` is used on every column so the
# resulting schemas match the original tables exactly (`BIGINT`, `DATE`, `DOUBLE`, `DECIMAL(5,2)`).

In [0]:
###################################################
# DEMO TABLES
###################################################

# ca_products (expected: 9 rows)
spark.sql(f"""
CREATE TABLE IF NOT EXISTS ca_products AS
SELECT
  CAST(productid   AS STRING)       AS productid,
  CAST(productname AS STRING)       AS productname,
  CAST(listprice   AS DECIMAL(5,2)) AS listprice
FROM read_files('{vol_path}/ca_products.csv', format => 'csv', header => 'true')
""")

# ca_customers (expected: 40 rows)
spark.sql(f"""
CREATE TABLE IF NOT EXISTS ca_customers AS
SELECT
  CAST(customerid   AS STRING) AS customerid,
  CAST(customername AS STRING) AS customername,
  CAST(city         AS STRING) AS city,
  CAST(province     AS STRING) AS province
FROM read_files('{vol_path}/ca_customers.csv', format => 'csv', header => 'true')
""")

# ca_opportunities (expected: 342 rows)
spark.sql(f"""
DROP TABLE IF EXISTS ca_opportunities;
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS ca_opportunities AS
SELECT
  CAST(opportunityid AS STRING) AS opportunityid,
  CAST(customerid    AS STRING) AS customerid,
  CAST(productid     AS STRING) AS productid,
  CAST(province      AS STRING) AS province,
  CAST(date          AS DATE)   AS date,
  CAST(amount        AS DOUBLE) AS amount,
  CAST(salesrep      AS STRING) AS salesrep,
  CAST(phase         AS STRING) AS phase
FROM read_files('{vol_path}/ca_opportunities.csv', format => 'csv', header => 'true')
""")

# ca_orders (expected: 15,536 rows)
spark.sql(f"""
DROP TABLE IF EXISTS ca_orders;
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS ca_orders AS
SELECT
  CAST(orderid    AS BIGINT) AS orderid,
  CAST(customerid AS STRING) AS customerid,
  CAST(productid  AS STRING) AS productid,
  CAST(orderdate  AS DATE)   AS orderdate,
  CAST(quantity   AS BIGINT) AS quantity,
  CAST(orderamt   AS DOUBLE) AS orderamt,
  CAST(salesrep   AS STRING) AS salesrep
FROM read_files('{vol_path}/ca_orders.csv', format => 'csv', header => 'true')
""")

# sales_targets (expected: 4 rows)
spark.sql(f"""
CREATE TABLE IF NOT EXISTS ca_sales_targets AS
SELECT *
FROM VALUES
  (
    2021,
    9000000.00,
    600,
    120000,
    DATE '2021-11-01',
    'Partial-year target (Q4 only; business launched mid-November).'
  ),
  (
    2022,
    80000000.00,
    5000,
    1500000,
    DATE '2022-01-05',
    'First full operating year; aggressive growth plan.'
  ),
  (
    2023,
    87000000.00,
    5400,
    1650000,
    DATE '2023-01-08',
    'Expected 8% YoY growth over 2022 actuals.'
  ),
  (
    2024,
    86000000.00,
    5300,
    1600000,
    DATE '2024-01-10',
    'Conservative hold-the-line plan after soft Q4 2023.'
  )
AS t (
  target_year,
  sales_target,
  orders_target,
  units_target,
  target_set_date,
  notes
)
""")

In [0]:
###################################################
# LAB TABLES
###################################################

# zip_codes
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {lab_schema}.zip_codes AS
SELECT
  CAST(zip_code  AS STRING)     AS zip_code,
  CAST(latitude  AS DOUBLE)     AS latitude,
  CAST(longitude AS DOUBLE)     AS longitude,
  CAST(zcta_id   AS STRING) AS zcta_id
FROM read_files('{vol_path}/demo_zip_codes.csv', format => 'csv', header => 'true')
""")

In [0]:
spark.sql(f'''
        CREATE TABLE IF NOT EXISTS {lab_schema}.nyctaxi_gold AS
        SELECT
        -- Original datetime columns
        tpep_pickup_datetime,
        tpep_dropoff_datetime,

        -- Pickup date values
        date(tpep_pickup_datetime) AS pickup_date,
        year(tpep_pickup_datetime) AS pickup_year,
        date_format(date_trunc('HOUR', tpep_pickup_datetime), 'HH:mm') AS pickup_hour,
        dayofweek(tpep_pickup_datetime) AS pickup_day_num,
        date_format(tpep_pickup_datetime, 'EEEE') AS pickup_day_name,

        -- Dropoff date values
        date(tpep_dropoff_datetime) AS dropoff_date,
        year(tpep_dropoff_datetime) AS dropoff_year,
        date_format(date_trunc('HOUR', tpep_dropoff_datetime), 'HH:mm') AS dropoff_hour,
        dayofweek(tpep_dropoff_datetime) AS dropoff_day_num,
        date_format(tpep_dropoff_datetime, 'EEEE') AS dropoff_day_name,

        -- Original measures
        trip_distance,
        fare_amount,

        -- Ride time in minutes
        timestampdiff(MINUTE, tpep_pickup_datetime, tpep_dropoff_datetime) AS ride_minutes,

        -- Cost measures
        round(try_divide(fare_amount, ride_minutes), 2) AS cost_per_minute,
        round(try_divide(fare_amount, trip_distance), 2) AS cost_per_mile,

        -- Average speed in MPH
        round(try_divide(trip_distance, ride_minutes/60.0), 2) AS avg_speed_mph,

        -- Pickup / dropoff zips with lat & lon
        lpad(CAST(pickup_zip AS STRING), 5, '0') AS pickup_zip,
        pickup.latitude  AS pickup_lat,
        pickup.longitude AS pickup_long,
        lpad(CAST(dropoff_zip AS STRING), 5, '0') AS dropoff_zip,
        dropoff.latitude  AS dropoff_lat,
        dropoff.longitude AS dropoff_long,

        -- Pickup to dropoff locations
        concat(pickup_zip,'-',dropoff_zip) AS pickup_to_dropoff

        FROM samples.nyctaxi.trips trips
        LEFT JOIN {lab_schema}.zip_codes pickup 
            ON trips.pickup_zip = pickup.zip_code
        LEFT JOIN {lab_schema}.zip_codes dropoff
            ON trips.dropoff_zip = dropoff.zip_code
        WHERE fare_amount > 0       
''')

In [0]:
display_config_values(
    [
        ("Your Catalog", my_catalog),
        ("Your Demonstration Schema", my_schema),
        ("Your Lab Schema", lab_schema)
    ]
)